In [ ]:
# ========== الجزء 1: استيراد المكتبات وقراءة البيانات ==========
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

plt.style.use('ggplot')
sns.set_palette("Set2")

# قراءة البيانات
df = pd.read_csv(r'C:\Users\Eng Hisham\Downloads\archive\StudentsPerformance.csv')

# ✨ تعديل أسماء الأعمدة (استبدال المسافات بشرطة سفلية)
df.columns = df.columns.str.replace(' ', '_')

# نظرة أولى
print("="*50)
print("📊 الشكل العام للبيانات:")
print(f"عدد الصفوف: {df.shape[0]} | عدد الأعمدة: {df.shape[1]}")
print("="*50)
print("\n📝 أسماء الأعمدة الجديدة:")
print(df.columns.tolist())

In [ ]:
# ========== الجزء 2: فحص جودة البيانات ==========
print("="*50)
print("🔍 فحص جودة البيانات:")
print("="*50)

print("\n❓ القيم المفقودة في كل عمود:")
print(df.isnull().sum())

print(f"\n🔄 عدد الصفوف المكررة: {df.duplicated().sum()}")

print("\n📈 إحصائيات الدرجات:")
print(df[['math_score', 'reading_score', 'writing_score']].describe())

In [ ]:
# ========== الجزء 3: تحليل البيانات ==========

# ===== تحليل 1: متوسط الدرجات حسب الجنس =====
gender_avg = df.groupby('gender')[['math_score', 'reading_score', 'writing_score']].mean().round(1)
print("\n👤 متوسط الدرجات حسب الجنس:")
print(gender_avg)

# ===== تحليل 2: تأثير الدورة التحضيرية =====
prep_avg = df.groupby('test_preparation_course')[['math_score', 'reading_score', 'writing_score']].mean().round(1)
print("\n📚 متوسط الدرجات حسب الدورة التحضيرية:")
print(prep_avg)

# ===== تحليل 3: تأثير مستوى تعليم الوالدين =====
parent_avg = df.groupby('parental_level_of_education')[['math_score', 'reading_score', 'writing_score']].mean().round(1)
print("\n👨‍👩‍👧 متوسط الدرجات حسب تعليم الوالدين:")
print(parent_avg)

# ===== تحليل 4: إضافة عمود إجمالي الدرجات والمعدل =====
df['total_score'] = df['math_score'] + df['reading_score'] + df['writing_score']
df['average_score'] = (df['total_score'] / 3).round(1)

# الطلاب المتفوقين (أعلى 5)
print("\n🏆 أعلى 5 طلاب درجات:")
print(df.nlargest(5, 'total_score')[['gender', 'math_score', 'reading_score', 'writing_score', 'total_score']])

# توزيع الطلاب حسب المستوى
def get_grade(avg):
    if avg >= 90:
        return 'A - ممتاز'
    elif avg >= 80:
        return 'B - جيد جداً'
    elif avg >= 70:
        return 'C - جيد'
    elif avg >= 60:
        return 'D - مقبول'
    else:
        return 'F - ضعيف'

df['grade'] = df['average_score'].apply(get_grade)
print("\n📊 توزيع الطلاب حسب التقدير:")
print(df['grade'].value_counts())

In [ ]:
# ========== الجزء 4: إنشاء الرسوم البيانية التفاعلية ==========

# ✨ تأكيد وجود الأعمدة المطلوبة (احتياطي)
if 'average_score' not in df.columns:
    df['total_score'] = df['math_score'] + df['reading_score'] + df['writing_score']
    df['average_score'] = (df['total_score'] / 3).round(1)
    
    def get_grade(avg):
        if avg >= 90:
            return 'A - ممتاز'
        elif avg >= 80:
            return 'B - جيد جداً'
        elif avg >= 70:
            return 'C - جيد'
        elif avg >= 60:
            return 'D - مقبول'
        else:
            return 'F - ضعيف'
    
    df['grade'] = df['average_score'].apply(get_grade)

print("\n" + "="*50)
print("📊 إنشاء الرسوم البيانية التفاعلية")
print("="*50)

# ====== الرسم البياني 1: Histogram - توزيع الدرجات ======
fig1 = px.histogram(
    df, 
    x='average_score',
    nbins=20,
    title='📊 توزيع متوسط درجات الطلاب',
    labels={'average_score': 'المعدل', 'count': 'عدد الطلاب'},
    color_discrete_sequence=['#4CAF50'],
    marginal='box'
)
fig1.update_layout(
    xaxis_title='المعدل',
    yaxis_title='عدد الطلاب',
    bargap=0.1
)
fig1.show()

# ====== الرسم البياني 2: Bar Chart - مقارنة الجنسين ======
gender_melted = df.melt(
    id_vars=['gender'],
    value_vars=['math_score', 'reading_score', 'writing_score'],
    var_name='المادة',
    value_name='الدرجة'
)

fig2 = px.bar(
    gender_melted.groupby(['gender', 'المادة'])['الدرجة'].mean().reset_index(),
    x='المادة',
    y='الدرجة',
    color='gender',
    barmode='group',
    title='👤 مقارنة متوسط درجات الذكور والإناث في المواد الثلاث',
    labels={'الدرجة': 'متوسط الدرجة', 'المادة': 'المادة'},
    color_discrete_map={'male': '#2196F3', 'female': '#E91E63'},
    text='الدرجة'
)
fig2.update_traces(texttemplate='%{text:.1f}', textposition='outside')
fig2.update_layout(
    xaxis_title='المادة',
    yaxis_title='متوسط الدرجة',
    yaxis_range=[0, 100]
)
fig2.show()

# ====== الرسم البياني 3: Pie Chart - توزيع التقديرات ======
grade_counts = df['grade'].value_counts().reset_index()
grade_counts.columns = ['التقدير', 'العدد']

grade_order = ['A - ممتاز', 'B - جيد جداً', 'C - جيد', 'D - مقبول', 'F - ضعيف']
grade_counts['التقدير'] = pd.Categorical(grade_counts['التقدير'], categories=grade_order, ordered=True)
grade_counts = grade_counts.sort_values('التقدير')

fig3 = px.pie(
    grade_counts,
    names='التقدير',
    values='العدد',
    title='📊 توزيع الطلاب حسب التقديرات',
    color='التقدير',
    color_discrete_map={
        'A - ممتاز': '#4CAF50',
        'B - جيد جداً': '#8BC34A',
        'C - جيد': '#FFC107',
        'D - مقبول': '#FF9800',
        'F - ضعيف': '#F44336'
    },hole=0.4
)
fig3.update_traces(textinfo='percent+label', pull=[0.1, 0, 0, 0, 0])
fig3.show()

# ====== الرسم البياني 4: Heatmap - العلاقة بين المواد ======
corr_matrix = df[['math_score', 'reading_score', 'writing_score']].corr().round(2)

fig4 = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect='auto',
    title='🔗 مصفوفة الارتباط بين المواد الثلاث',
    labels={'x': 'المادة', 'y': 'المادة', 'color': 'معامل الارتباط'},
    color_continuous_scale='RdYlGn'
)
fig4.update_layout(
    xaxis_side='bottom'
)
fig4.show()

print("\n✅ تم إنشاء جميع الرسوم البيانية بنجاح!")